<a href="https://colab.research.google.com/github/navneetkrc/Deep-Learning-Experiments-implemented-using-Google-Colab/blob/master/ModernBERT_fine_tuning_example_UnfoldAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ModernBERT fine-tuning example

A practical guide to fine-tuning ModernBERT for sentiment analysis. This notebook demonstrates how to adapt ModernBERT for Bulgarian sentiment analysis using a GPU-optimized implementation.

Run it on a **free** Tesla T4 Colab instance!  

To begin, click "*Runtime*" → "*Run all*" in the menu above.

---



[![UnfoldAI](https://unfoldai.com/storage/2023/12/unfoldai-logo-b.png)](https://unfoldai.com)

_Created with ❤️ by Simeon Emanuilov_

Software Engineer & Ph.D. candidate | Specializing in ML/DL system development & applying AI to solve real-world business problems.

[GitHub](https://github.com/s-emanuilov) | [Substack](https://substack.com/@emanuilov) | [Twitter](https://x.com/s_emanuilov) | [LinkedIn](https://www.linkedin.com/in/simeon-emanuilov/)

---

In this tutorial, you'll understand the essential steps of fine-tuning ModernBERT:

- Configuring ModernBERT for your specific use case;
- Processing and preparing custom datasets;
- Training the model;
- Evaluating performance and deploying results.

Let's dive in and start fine-tuning! 🎯

---

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git datasets

In [ ]:
# Import required PyTorch libraries for deep learning operations and model components
# - torch: Main PyTorch library for tensor operations and neural networks
# - AutoTokenizer: For converting text into tokens that the model can understand
# - AutoModelForSequenceClassification: Pre-built model architecture for classification tasks
# - Dataset, DataLoader: PyTorch utilities for handling data efficiently
# - load_dataset: HuggingFace utility to load datasets from their hub
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

In [ ]:
# Custom Dataset class for handling text data and preparing it for model training
# This class inherits from PyTorch's Dataset class and implements required methods:
# - __init__: Initializes the dataset by tokenizing text and preparing labels
# - __getitem__: Returns a single data item with its features and label
# - __len__: Returns the total number of items in the dataset
# The max_length parameter controls the maximum number of tokens per text sample
class TextDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.encodings = tokenizer(data['text'], truncation=True, padding=True,
                                 max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(data['label'])

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Load dataset
dataset = load_dataset("DGurgurov/bulgarian_sa")

# Initialize tokenizer and model
model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Create datasets
train_dataset = TextDataset(dataset['train'], tokenizer)
val_dataset = TextDataset(dataset['validation'], tokenizer)
test_dataset = TextDataset(dataset['test'], tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/929 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/489k [00:00<?, ?B/s]

dev.csv:   0%|          | 0.00/68.5k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/145k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Training loop
num_epochs = 7

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()

            predictions = torch.argmax(outputs.logits, dim=1)
            correct += (predictions == batch['labels']).sum().item()
            total += batch['labels'].size(0)

    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    accuracy = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Average training loss: {avg_train_loss:.4f}")
    print(f"Average validation loss: {avg_val_loss:.4f}")
    print(f"Validation accuracy: {accuracy:.4f}")

# Final test set evaluation
model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        test_loss += outputs.loss.item()

        predictions = torch.argmax(outputs.logits, dim=1)
        correct += (predictions == batch['labels']).sum().item()
        total += batch['labels'].size(0)

test_accuracy = correct / total
print(f"\nFinal Test Accuracy: {test_accuracy:.4f}")

Epoch 1/7
Average training loss: 0.3253
Average validation loss: 0.1898
Validation accuracy: 0.9236
Epoch 2/7
Average training loss: 0.1766
Average validation loss: 0.2263
Validation accuracy: 0.9081
Epoch 3/7
Average training loss: 0.0979
Average validation loss: 0.2377
Validation accuracy: 0.9368
Epoch 4/7
Average training loss: 0.0510
Average validation loss: 0.2032
Validation accuracy: 0.9511
Epoch 5/7
Average training loss: 0.0371
Average validation loss: 0.1605
Validation accuracy: 0.9523
Epoch 6/7
Average training loss: 0.0543
Average validation loss: 0.2214
Validation accuracy: 0.9499
Epoch 7/7
Average training loss: 0.0193
Average validation loss: 0.2246
Validation accuracy: 0.9368

Final Test Accuracy: 0.9283


In [ ]:
# Save the model
model.save_pretrained('./modernbert_bulgarian_sa')
tokenizer.save_pretrained('./modernbert_bulgarian_sa')

('./modernbert_bulgarian_sa/tokenizer_config.json',
 './modernbert_bulgarian_sa/special_tokens_map.json',
 './modernbert_bulgarian_sa/tokenizer.json')